In [1]:
import pandas as pd 
vitalsign = pd.read_csv('vitalsign.csv')

In [2]:
import pandas as pd
import numpy as np

# =========================================================
# Helper Functions
# =========================================================

def normalize_sbp(x):
    """
    Normalize systolic blood pressure values
    Example:
    1234 -> 123.4
    12080 -> 120.8
    151103 -> 151.103 -> 151.1
    """
    if pd.isna(x):
        return np.nan

    while x > 300:
        x = x / 10

    if x < 60 or x > 300:
        return np.nan

    return x


def normalize_dbp(x):
    """
    Normalize diastolic blood pressure values
    Example:
    74810 -> 74.81
    97100 -> 97.1
    """
    if pd.isna(x):
        return np.nan

    while x > 200:
        x = x / 10

    if x < 30 or x > 200:
        return np.nan

    return x


# =========================================================
# Optional: Convert zeros to NULL first
# =========================================================

cols = [
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp"
]

for col in cols:
    vitalsign.loc[vitalsign[col] == 0, col] = np.nan


# =========================================================
# Temperature
# =========================================================

# Example: 986 -> 98.6
vitalsign.loc[
    vitalsign["temperature"] > 200,
    "temperature"
] = vitalsign["temperature"] / 10

# Final clinical validation
vitalsign.loc[
    (vitalsign["temperature"] < 85) |
    (vitalsign["temperature"] > 110),
    "temperature"
] = np.nan


# =========================================================
# Heart Rate
# =========================================================

vitalsign.loc[
    (vitalsign["heartrate"] < 25) |
    (vitalsign["heartrate"] > 250),
    "heartrate"
] = np.nan


# =========================================================
# Respiratory Rate
# =========================================================

vitalsign.loc[
    (vitalsign["resprate"] < 5) |
    (vitalsign["resprate"] > 60),
    "resprate"
] = np.nan


# =========================================================
# O2 Saturation
# =========================================================

# Cap upper bound
vitalsign.loc[
    vitalsign["o2sat"] > 100,
    "o2sat"
] = 100

# Remove impossible low values
vitalsign.loc[
    vitalsign["o2sat"] < 50,
    "o2sat"
] = np.nan


# =========================================================
# SBP (Improved normalization)
# =========================================================

vitalsign["sbp"] = vitalsign["sbp"].apply(normalize_sbp)


# =========================================================
# DBP (Improved normalization)
# =========================================================

vitalsign["dbp"] = vitalsign["dbp"].apply(normalize_dbp)


print("Vital signs cleaning completed successfully.")
#save to new csv file
vitalsign.to_csv('vitalsign_fixed.csv', index=False)

Vital signs cleaning completed successfully.


In [3]:
df=pd.read_csv('vitalsign_fixed.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1564610 entries, 0 to 1564609
Data columns (total 11 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   subject_id   1564610 non-null  int64  
 1   stay_id      1564610 non-null  int64  
 2   charttime    1564610 non-null  str    
 3   temperature  995259 non-null   float64
 4   heartrate    1494164 non-null  float64
 5   resprate     1474648 non-null  float64
 6   o2sat        1427960 non-null  float64
 7   sbp          1480618 non-null  float64
 8   dbp          1481619 non-null  float64
 9   rhythm       59650 non-null    str    
 10  pain         1121344 non-null  str    
dtypes: float64(6), int64(2), str(3)
memory usage: 131.3 MB


In [5]:
# Insert to database
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="asclena_db",
    user="postgres",
    password="admin123"
)

cur = conn.cursor()

In [6]:
#insert data into database
query = """
INSERT INTO asclena.vital_sign (
    subject_id, stay_id, charttime,
    temperature, heartrate, resprate,
    o2sat, sbp, dbp, rhythm, pain
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

batch_size = 5000
failed_rows = []

for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size]

    try:
        cur.executemany(query, batch.values.tolist())
        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Batch failed at {i}: {e}")

        # log bad batch for debugging
        failed_rows.append(i)

In [8]:
batch

,subject_id,stay_id,charttime,temperature,heartrate,resprate,o2sat,sbp,dbp,rhythm,pain
1560000,19970078,31482607,2196-03-31 07:37:00,97.5,71.0,16.0,99.0,101.0,65.0,NaN,NaN
1560001,19970078,31482607,2196-03-31 11:30:00,97.8,59.0,16.0,99.0,137.0,90.0,NaN,0
1560002,19970078,32140074,2192-09-15 00:15:00,NaN,58.0,18.0,NaN,NaN,NaN,NaN,0
1560003,19970078,32140074,2192-09-15 01:07:00,97.5,57.0,15.0,NaN,160.0,96.0,NaN,0
1560004,19970078,33520182,2198-01-14 15:29:00,98.2,81.0,20.0,100.0,149.0,71.0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...
1564605,19999828,32917002,2149-01-08 17:10:00,98.1,109.0,15.0,96.0,111.0,78.0,NaN,NaN
1564606,19999914,32002659,2158-12-24 11:43:00,99.5,81.0,10.0,100.0,93.0,55.0,NaN,0
1564607,19999987,34731548,2145-11-02 19:40:00,NaN,112.0,18.0,NaN,118.0,83.0,NaN,NaN
1564608,19999987,34731548,2145-11-02 20:11:00,NaN,111.0,18.0,NaN,123.0,82.0,NaN,unable
